# 🤖 Document AI – Chat with Your PDF Documents

An AI-powered PDF question-answering application built with Python, Google Colab, Streamlit, ChromaDB, Sentence Transformers, and Google's Gemini API.

The application uses **Retrieval-Augmented Generation (RAG)** to retrieve relevant information from uploaded PDF documents and generate context-aware answers using a Large Language Model (LLM).

## 🚀 Project Objectives

- Extract text from PDF documents.
- Divide documents into smaller text chunks.
- Generate semantic embeddings from text.
- Store and search embeddings using ChromaDB.
- Retrieve relevant document information based on user questions.
- Generate context-aware answers using Google's Gemini API.
- Display generated answers progressively.
- Build an interactive interface using Streamlit.

## 🛠️ Technologies Used

| Technology | Purpose |
|---|---|
| Python | Core programming language |
| Google Colab | Development environment |
| PyPDF | PDF text extraction |
| Sentence Transformers | Embedding generation |
| ChromaDB | Vector database |
| Google Gemini API | Answer generation |
| Streamlit | Interactive interface |
| Pytest | Automated testing |

## 🧠 RAG Architecture

The application follows a Retrieval-Augmented Generation architecture.

### How It Works

1. **Document Loading:** Extract text from uploaded PDF documents.
2. **Text Splitting:** Divide extracted text into smaller chunks.
3. **Embedding Generation:** Convert text chunks into numerical vector representations.
4. **Vector Storage:** Store embeddings and document content in ChromaDB.
5. **Question Processing:** Convert the user's question into an embedding.
6. **Semantic Retrieval:** Search ChromaDB for relevant document chunks.
7. **Context Preparation:** Combine retrieved information with the user's question.
8. **Answer Generation:** Send the context and question to Gemini.
9. **Response Streaming:** Display the generated answer progressively.

## 🎯 Expected Outcome

A user-friendly AI application that allows users to upload PDF documents and ask questions about their contents, receiving answers grounded in the uploaded documents.

## 📦 Step 2: Install Project Dependencies

Before building the application, we need to install the Python libraries required by our RAG pipeline.

### Libraries

- **ChromaDB:** Stores document embeddings and supports semantic search.
- **PyPDF:** Extracts text from PDF documents.
- **Sentence Transformers:** Converts text into numerical embeddings.
- **Google GenAI:** Connects our application to Google's Gemini API.
- **Python-dotenv:** Helps manage environment variables.
- **Pytest:** Supports automated testing.

We will install these dependencies in the Google Colab runtime.

In [2]:

%pip install -q chromadb pypdf sentence-transformers google-genai python-dotenv pytest


## ✅ Step 3: Verify Installed Libraries

After installing the dependencies, we need to verify that Python can successfully import each library.

Importing a library makes its functionality available to our Python program.

If all imports work without errors, our development environment is ready for the next stage.

In [3]:

import chromadb
import pypdf
import sentence_transformers
from google import genai
import dotenv
import pytest

print("All packages imported successfully!")


All packages imported successfully!


## 📄 Step 4: Upload PDF Documents

Before our chatbot can answer questions about a document, it needs access to that document.

Google Colab provides a file-upload utility that allows us to upload files directly from our computer into the notebook's temporary runtime environment.

### How it works

1. The user selects a PDF file from their computer.
2. Google Colab uploads the file into its runtime.
3. Python receives the uploaded file name and its contents.
4. We can then use PyPDF to extract the document's text.

### Important note

Files uploaded directly into Colab's runtime are temporary. They may disappear when the runtime is disconnected or reset.

In [4]:

from google.colab import files

uploaded_files = files.upload()

print("\nUploaded files:")

for filename in uploaded_files.keys():
    print(filename)


Saving VLSI_Assignment_RISC_and_CISC_Processors.pdf to VLSI_Assignment_RISC_and_CISC_Processors.pdf

Uploaded files:
VLSI_Assignment_RISC_and_CISC_Processors.pdf


## 📝 Step 5: Extract Text from PDF Documents

Uploading a PDF does not automatically give our application its readable text.

We need to extract the text using the **PyPDF** library.

### What is PyPDF?

PyPDF is a Python library for working with PDF files. It allows us to read PDF documents, access their pages, and extract text.

### How text extraction works

- `PdfReader()` opens the PDF document.
- `reader.pages` gives access to its pages.
- `page.extract_text()` extracts text from an individual page.
- We combine the extracted text from all pages into one string.

The extracted text will become the input for our next component: the text splitter.

In [5]:

from pypdf import PdfReader

# Get the uploaded PDF filename
pdf_filename = list(uploaded_files.keys())[0]

# Open the PDF
reader = PdfReader(pdf_filename)

# Extract text from every page
extracted_text = ""

for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""
    extracted_text += f"\nPage {page_number}\n{page_text}"

# Display extraction results
print("PDF loaded successfully!")
print("Total pages:", len(reader.pages))
print("Extracted characters:", len(extracted_text))

print("\nFirst 500 characters:\n")
print(extracted_text[:500])


PDF loaded successfully!
Total pages: 7
Extracted characters: 13134

First 500 characters:


Page 1
Page 1
RISC AND CISC PROCESSORS
A  processor,  also  known  as  a  central  processing  unit  (CPU),  is  the  main  component  of  a 
computer that executes instructions and controls many of the system's operations. A processor  
uses an instruction set architecture (ISA), which defines the instructions that software can use  
and how those instructions are represented and executed by the hardware.
Two  important  approaches  to  processor  instruction-set  design  are  Reduced  Instruc


## ✂️ Step 6: Text Chunking

### What is text chunking?

Text chunking is the process of dividing a large document into smaller pieces of text called **chunks**.

Why do we need it?

Imagine uploading a 100-page PDF document. Sending the entire document to an AI model whenever a user asks a question can be inefficient because:

- Large documents may exceed the model's context limits.
- Processing unnecessary information increases cost and latency.
- Searching smaller sections makes information retrieval more efficient.

Instead, we divide the document into smaller chunks.

### Chunk Size

Chunk size determines how many characters each chunk contains.

For example, a chunk size of 1,000 means each chunk contains approximately 1,000 characters.

### Chunk Overlap

Chunk overlap determines how many characters are shared between consecutive chunks.

For example:

- Chunk size: 1,000 characters
- Overlap: 200 characters
- Next chunk starts 800 characters after the previous chunk starts.

Overlap helps preserve context when a sentence or paragraph crosses a chunk boundary.

### Our Configuration

| Parameter | Value |
|---|---:|
| Chunk size | 1,000 |
| Overlap | 200 |
| Step size | 800 |

We will use these values for our first implementation.

In [6]:

def split_text(
    text: str,
    chunk_size: int = 1000,
    overlap: int = 200
) -> list[str]:
    """
    Split text into smaller overlapping chunks.
    """

    if chunk_size <= 0:
        raise ValueError("Chunk size must be greater than zero.")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError(
            "Overlap must be non-negative and smaller than chunk size."
        )

    if not text or not text.strip():
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


In [7]:

chunks = split_text(
    extracted_text,
    chunk_size=1000,
    overlap=200
)

print("Document successfully split!")
print("Total characters:", len(extracted_text))
print("Total chunks:", len(chunks))

for index, chunk in enumerate(chunks[:3], start=1):
    print(f"\n--- Chunk {index} ---")
    print(chunk[:200])


Document successfully split!
Total characters: 13134
Total chunks: 17

--- Chunk 1 ---
Page 1
Page 1
RISC AND CISC PROCESSORS
A  processor,  also  known  as  a  central  processing  unit  (CPU),  is  the  main  component  of  a 
computer that executes instructions and controls many of t

--- Chunk 2 ---
complex operations. [1], [2]
These architectures are important in Very Large Scale Integration (VLSI) because instruction-set  
design  influences  processor  datapaths,  control  logic,  registers,  

--- Chunk 3 ---
rk on data held in CPU registers. Separate load and  
store instructions transfer data between memory and registers. RISC designs also commonly use  
a large number of general-purpose registers and re


## 🧠 Step 7: Generate Text Embeddings

### What are embeddings?

Embeddings are numerical representations of text. An embedding model converts a sentence, paragraph, or document into a list of numbers called a **vector**.

Computers can use these vectors to compare the meaning of different pieces of text.

### Why do we need embeddings?

Imagine our PDF contains this sentence:

"RISC processors use a small set of simple instructions."

A user asks:

"What is the instruction design of RISC architecture?"

The question and the original sentence use different words, but they have similar meanings.

A keyword search might struggle to recognize this relationship. Semantic search uses embeddings to find text with similar meanings.

### What is Sentence Transformers?

Sentence Transformers is a Python library that provides pretrained models for converting text into embeddings.

We will use:

`all-MiniLM-L6-v2`

This model:
- Converts text into numerical vectors.
- Produces vectors with 384 dimensions.
- Is relatively lightweight.
- Is suitable for semantic similarity and retrieval tasks.

### How our embedding process works

1. Take the text chunks generated in Step 6.
2. Pass each chunk to the embedding model.
3. Convert each chunk into a numerical vector.
4. Keep the original text associated with its vector.

These vectors will be stored in ChromaDB in our next step.

In [9]:

from sentence_transformers import SentenceTransformer

# Load the pretrained embedding model
MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded successfully!")
print("Model:", MODEL_NAME)
print(
    "Embedding dimensions:",
    embedding_model.get_sentence_embedding_dimension()
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!
Model: all-MiniLM-L6-v2
Embedding dimensions: 384


/tmp/ipykernel_1687/3288342286.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


## 🔢 Step 8: Generate Embeddings for Document Chunks

Now that we have loaded our embedding model, we can convert our document chunks into numerical vectors.

The `encode()` method processes text and generates embeddings.

### Important parameters

- `chunks`: The list of text chunks we created in Step 6.
- `convert_to_numpy=True`: Returns the embeddings as NumPy arrays.
- `show_progress_bar=True`: Displays progress while generating embeddings.

### Expected result

Our model produces a vector containing 384 numerical values for each chunk.

For example, if our document has 17 chunks, the embedding array will have this shape:

`(17, 384)`

This means:
- 17 rows represent our 17 text chunks.
- 384 columns represent the numerical dimensions of each embedding.

We will store these embeddings in ChromaDB in the next stage.

In [10]:

# Generate embeddings for all document chunks
chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

# Display embedding information
print("Embeddings generated successfully!")

print("Number of chunks:", len(chunks))
print("Embedding shape:", chunk_embeddings.shape)
print("Vector dimensions:", chunk_embeddings.shape[1])

print("\nFirst 10 values of the first embedding:")
print(chunk_embeddings[0][:10])


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully!
Number of chunks: 17
Embedding shape: (17, 384)
Vector dimensions: 384

First 10 values of the first embedding:
[-0.01261634  0.0157543  -0.02372114 -0.04088796 -0.00697863 -0.07134268
 -0.01335918  0.07029117  0.07801476 -0.00155401]


## 🗄️ Step 9: Store Embeddings Using ChromaDB

### What is ChromaDB?

ChromaDB is an open-source vector database designed to store, manage, and retrieve vector embeddings.

In our RAG application, ChromaDB stores three important things:

1. **Embeddings:** Numerical representations of our text chunks.
2. **Documents:** The original text associated with each embedding.
3. **Metadata:** Additional information about each chunk, such as its source filename and chunk number.

### What is a Collection?

A collection in ChromaDB is similar to a table in a traditional database.

It groups related documents and their embeddings together.

For our project, we'll create a collection called `document_collection`.

### How will we store our data?

For each chunk, we'll create:

- A unique ID.
- Its original text.
- Its embedding vector.
- Metadata containing the PDF filename and chunk number.

We'll use `upsert()` to insert new records or update existing records with the same IDs.

We'll configure the collection to use cosine distance for similarity comparisons.

In [11]:

import chromadb

# Create a persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Create or retrieve our collection
collection = chroma_client.get_or_create_collection(
    name="document_collection",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB initialized successfully!")
print("Collection name:", collection.name)


ChromaDB initialized successfully!
Collection name: document_collection


In [12]:

# Create unique IDs for each chunk
chunk_ids = [
    f"chunk_{i}"
    for i in range(len(chunks))
]

# Create metadata for each chunk
chunk_metadata = [
    {
        "source": pdf_filename,
        "chunk_index": i
    }
    for i in range(len(chunks))
]

# Store chunks, embeddings, IDs, and metadata
collection.upsert(
    ids=chunk_ids,
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    metadatas=chunk_metadata
)

print("Document chunks stored successfully!")
print("Total stored chunks:", collection.count())


Document chunks stored successfully!
Total stored chunks: 17


In [13]:

# Retrieve the first three stored records
stored_data = collection.get(
    limit=3,
    include=["documents", "metadatas"]
)

for i in range(len(stored_data["ids"])):
    print(f"\n--- Stored Record {i + 1} ---")
    print("ID:", stored_data["ids"][i])
    print("Metadata:", stored_data["metadatas"][i])
    print("Document:", stored_data["documents"][i][:200])



--- Stored Record 1 ---
ID: chunk_0
Metadata: {'chunk_index': 0, 'source': 'VLSI_Assignment_RISC_and_CISC_Processors.pdf'}
Document: Page 1
Page 1
RISC AND CISC PROCESSORS
A  processor,  also  known  as  a  central  processing  unit  (CPU),  is  the  main  component  of  a 
computer that executes instructions and controls many of t

--- Stored Record 2 ---
ID: chunk_1
Metadata: {'source': 'VLSI_Assignment_RISC_and_CISC_Processors.pdf', 'chunk_index': 1}
Document: complex operations. [1], [2]
These architectures are important in Very Large Scale Integration (VLSI) because instruction-set  
design  influences  processor  datapaths,  control  logic,  registers,  

--- Stored Record 3 ---
ID: chunk_2
Metadata: {'chunk_index': 2, 'source': 'VLSI_Assignment_RISC_and_CISC_Processors.pdf'}
Document: rk on data held in CPU registers. Separate load and  
store instructions transfer data between memory and registers. RISC designs also commonly use  
a large number of general-purpose registers an

Step 12: Semantic Search
Semantic search finds information based on the meaning of a question, not just matching exact words.

In [14]:

def search_documents(query: str, n_results: int = 3) -> dict:
    """
    Search the document collection using semantic similarity.
    """

    if not query or not query.strip():
        raise ValueError("Search query cannot be empty.")

    # Convert the question into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    # Search ChromaDB
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=min(n_results, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    return results


print("Semantic search function created successfully!")


Semantic search function created successfully!


In [15]:

question = "What are the main characteristics of RISC processors?"

search_results = search_documents(
    query=question,
    n_results=3
)

print("Question:", question)

for i, document in enumerate(search_results["documents"][0]):
    print(f"\n--- Result {i + 1} ---")
    print("Distance:", search_results["distances"][0][i])
    print("Metadata:", search_results["metadatas"][0][i])
    print("Content:")
    print(document)


Question: What are the main characteristics of RISC processors?

--- Result 1 ---
Distance: 0.30296391248703003
Metadata: {'chunk_index': 14, 'source': 'VLSI_Assignment_RISC_and_CISC_Processors.pdf'}
Content:
system design.
In VLSI design, the choice of architecture is one part of a larger engineering decision. Designers  
must also consider the target application, performance requirements, energy budget, silicon area,  
cost, software ecosystem, and compatibility.
8. Conclusion
RISC and CISC are two major approaches to designing processor instruction sets. RISC uses a  
relatively small and regular set of simple instructions, which can support simpler decoding and  
efficient pipelining. CISC provides a larger and more complex instruction set, allowing some  
tasks to be represented with fewer instructions and supporting rich addressing modes.
RISC architectures such as ARM and RISC-V are common in mobile, embedded, IoT, and other  
systems,  while  CISC  architectures  such  as  x86-

Step 13: Prepare Context for Gemini

What is context preparation?
Remember, our project is a RAG (Retrieval-Augmented Generation) application.

RAG combines two important things:
1. Retrieval: Find relevant information from your PDF using ChromaDB.
2. Generation: Give that information to an AI model (Gemini) so it can generate an answer.
Before sending information to Gemini, we need to prepare the retrieved document chunks into a single context.

In [16]:

def prepare_context(search_results: dict) -> str:
    """
    Combine retrieved document chunks into one context string.
    """

    documents = search_results["documents"][0]

    if not documents:
        return "No relevant information was found in the document."

    context_parts = []

    for index, document in enumerate(documents, start=1):
        context_parts.append(
            f"Document Chunk {index}:\n{document}"
        )

    context = "\n\n".join(context_parts)

    return context


print("Context preparation function created successfully!")


Context preparation function created successfully!


In [17]:

question = "What are the main characteristics of RISC processors?"

search_results = search_documents(
    query=question,
    n_results=3
)

context = prepare_context(search_results)

print("QUESTION:")
print(question)

print("\nRETRIEVED CONTEXT:")
print(context)


QUESTION:
What are the main characteristics of RISC processors?

RETRIEVED CONTEXT:
Document Chunk 1:
system design.
In VLSI design, the choice of architecture is one part of a larger engineering decision. Designers  
must also consider the target application, performance requirements, energy budget, silicon area,  
cost, software ecosystem, and compatibility.
8. Conclusion
RISC and CISC are two major approaches to designing processor instruction sets. RISC uses a  
relatively small and regular set of simple instructions, which can support simpler decoding and  
efficient pipelining. CISC provides a larger and more complex instruction set, allowing some  
tasks to be represented with fewer instructions and supporting rich addressing modes.
RISC architectures such as ARM and RISC-V are common in mobile, embedded, IoT, and other  
systems,  while  CISC  architectures  such  as  x86-64  are  widely  used  in  desktops,  laptops,  
workstations, and servers. Neither approach is universally

Step 14: Connect the Gemini API

What is the Gemini API?
So far, ChromaDB finds relevant information, but it does not generate a natural-language answer.

Gemini is the language model that will:
1. Receive the user's question.
2. Receive the context retrieved from ChromaDB.
3. Understand the supplied information.
4. Generate an answer based on that information.
Important: We will keep your API key private using Google Colab Secrets. Never write your actual API key directly in notebook code.

In [20]:

from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

MODEL_NAME = "gemini-3.5-flash-lite"

print("Gemini client initialized successfully!")
print("Model:", MODEL_NAME)


Gemini client initialized successfully!
Model: gemini-3.5-flash-lite


In [21]:

response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Explain what a RISC processor is in two simple sentences."
)

print("Gemini response:")
print(response.text)


Gemini response:
A RISC (Reduced Instruction Set Computer) processor uses a small, simple set of instructions that it can execute very quickly. By focusing on only the most essential commands, it achieves high efficiency and consumes less power than traditional processors.


Step 15: Build the RAG Answer Generation Function

How will it work?
When a user asks a question:
1. The question goes to ChromaDB.
2. ChromaDB retrieves the three most relevant PDF chunks.
3. We combine those chunks into context.
4. We send the question and context to Gemini.
5. Gemini generates an answer based on the PDF.
One important rule: Gemini should answer using the provided document context. If the document doesn't contain the answer, it should say so instead of inventing information.

In [22]:

def generate_answer(question: str, n_results: int = 3) -> dict:
    """
    Retrieve relevant document chunks and generate
    an answer using Gemini.
    """

    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    # Step 1: Retrieve relevant document chunks
    search_results = search_documents(
        query=question,
        n_results=n_results
    )

    # Step 2: Prepare context
    context = prepare_context(search_results)

    # Step 3: Create the prompt
    prompt = f"""
You are a helpful document question-answering assistant.

Answer the user's question using ONLY the provided context.

If the context does not contain enough information,
clearly say that the answer was not found in the document.

Do not invent facts.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    # Step 4: Generate the answer
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return {
        "question": question,
        "answer": response.text,
        "context": context,
        "sources": search_results["metadatas"][0]
    }


print("RAG answer generation function created successfully!")


RAG answer generation function created successfully!


In [23]:

question = "What are the main characteristics of RISC processors?"

result = generate_answer(question)

print("QUESTION:")
print(result["question"])

print("\nGENERATED ANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)


QUESTION:
What are the main characteristics of RISC processors?

GENERATED ANSWER:
Based on the provided document, the main characteristics of RISC (Reduced Instruction Set Computer) processors include:

* **Small and simple instruction set:** It uses a comparatively small, simple, and regular set of instructions where most instructions perform one basic operation (such as adding values, comparing values, or moving data between registers).
* **Load/store design:** Arithmetic and logical operations typically work on data held in CPU registers, while separate load and store instructions are used to transfer data between memory and registers.
* **Large number of general-purpose registers:** RISC designs commonly use many general-purpose registers.
* **Regular instruction formats:** This feature, along with simple instructions, supports simpler decoding and efficient pipelining.
* **Relative simplicity and regularity:** The instruction set is simple and regular, meaning complex tasks are c

Step 16: Add Streaming Responses
What is streaming?
Normally, Gemini generates the complete answer before returning it to your application.

With streaming, Gemini sends the answer in smaller pieces (chunks) as it generates them.

For example:
- Without streaming: You wait, then see the entire answer.
- With streaming: You see the first words, then more words appear until the answer is complete.
This makes our chatbot feel faster and more interactive.
How streaming works
The Google Gen AI SDK provides generate_content_stream().

We will:
1. Retrieve relevant chunks from ChromaDB.
2. Prepare the context.
3. Send the prompt to Gemini.
4. Display each response chunk as it arrives

In [24]:

def generate_answer_stream(question: str, n_results: int = 3):
    """
    Retrieve relevant document chunks and stream
    Gemini's generated answer.
    """

    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    # Step 1: Retrieve relevant document chunks
    search_results = search_documents(
        query=question,
        n_results=n_results
    )

    # Step 2: Prepare context
    context = prepare_context(search_results)

    # Step 3: Create the prompt
    prompt = f"""
You are a helpful document question-answering assistant.

Answer the user's question using ONLY the provided context.

If the context does not contain enough information,
clearly say that the answer was not found in the document.

Do not invent facts.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    # Step 4: Stream Gemini's response
    response_stream = client.models.generate_content_stream(
        model=MODEL_NAME,
        contents=prompt
    )

    print("ANSWER:\n")

    for chunk in response_stream:
        if chunk.text:
            print(chunk.text, end="", flush=True)

    print("\n\nAnswer generation completed!")

    # Return retrieved sources
    return search_results["metadatas"][0]


print("Streaming function created successfully!")


Streaming function created successfully!


In [25]:

question = "What are the main differences between RISC and CISC processors?"

sources = generate_answer_stream(question)

print("\nSOURCES:")

for source in sources:
    print(source)


ANSWER:

Based on the provided document, the main traditional differences between RISC (Reduced Instruction Set Computer) and CISC (Complex Instruction Set Computer) processors are:

* **Instruction Set:** RISC usually has a smaller and more regular instruction set, whereas CISC provides a larger and more varied instruction set.
* **Instruction Complexity:** RISC instructions generally perform simple operations, while CISC includes some instructions that perform complex operations.
* **Instruction Format:** RISC often uses fixed-length or a small number of regular formats, whereas CISC often uses variable-length or more varied formats.
* **Memory Access:** RISC commonly uses load/store instructions for memory access, while CISC may allow instructions to operate directly on memory operands.
* **Registers:** RISC often provides many general-purpose registers, whereas CISC historically used fewer general-purpose registers (though modern designs vary).
* **Control Unit:** RISC is often sim

Step 17: Add Conversation History
What is conversation history?
Currently, our chatbot treats every question independently.

For example:

Question 1: What is RISC?

Answer: RISC stands for Reduced Instruction Set Computer...

Question 2: What are its advantages?

The problem is that the chatbot may not understand what "its" refers to because it doesn't remember the previous question.

Conversation history solves this problem by keeping track of previous questions and answers.

Our chatbot will maintain a list containing messages such as:
- User questions.
- Assistant answers.
When a new question arrives, we will send the previous conversation along with the newly retrieved PDF context.

In [26]:

conversation_history = []

print("Conversation history initialized!")
print("Number of messages:", len(conversation_history))


Conversation history initialized!
Number of messages: 0


In [27]:

def chat_with_document(question: str, n_results: int = 3) -> str:
    """
    Chat with the PDF using RAG and conversation history.
    """

    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    # Step 1: Retrieve relevant document chunks
    search_results = search_documents(
        query=question,
        n_results=n_results
    )

    # Step 2: Prepare context
    context = prepare_context(search_results)

    # Step 3: Prepare conversation history
    history = ""

    for message in conversation_history:
        history += (
            f"{message['role'].upper()}: "
            f"{message['content']}\n"
        )

    # Step 4: Create the prompt
    prompt = f"""
You are a helpful PDF question-answering assistant.

Answer questions using the provided document context.
Use conversation history to understand follow-up questions.

If the answer is not available in the document,
clearly explain that you could not find it.

Do not invent information.

CONVERSATION HISTORY:
{history}

DOCUMENT CONTEXT:
{context}

CURRENT QUESTION:
{question}

ANSWER:
"""

    # Step 5: Generate an answer
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    answer = response.text

    # Step 6: Save the conversation
    conversation_history.append({
        "role": "user",
        "content": question
    })

    conversation_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer


print("Chatbot function created successfully!")


Chatbot function created successfully!


In [28]:

# First question
answer1 = chat_with_document(
    "What is a RISC processor?"
)

print("USER: What is a RISC processor?")
print("\nASSISTANT:")
print(answer1)


# Follow-up question
answer2 = chat_with_document(
    "What are its main advantages?"
)

print("\n" + "=" * 50)
print("USER: What are its main advantages?")
print("\nASSISTANT:")
print(answer2)


USER: What is a RISC processor?

ASSISTANT:
Based on the provided document, RISC stands for Reduced Instruction Set Computer. It is a processor design approach that uses a comparatively small and simple set of instructions. Most RISC instructions are designed to perform one basic operation, such as adding two values, comparing values, or moving data between registers. 

Key features of RISC processors include:
* **Load/store design:** Arithmetic and logical operations usually work on data held in CPU registers, while separate load and store instructions transfer data between memory and registers.
* **Large number of general-purpose registers** and regular instruction formats, which help make instruction decoding and pipelining simpler.
* **Relative simplicity and regularity** in its instruction set rather than lacking power, where complex tasks are completed by combining several simple instructions.

USER: What are its main advantages?

ASSISTANT:
Based on the provided document, the ma

In [29]:

print("CONVERSATION HISTORY")

for message in conversation_history:
    print(f"\n{message['role'].upper()}:")
    print(message["content"])


CONVERSATION HISTORY

USER:
What is a RISC processor?

ASSISTANT:
Based on the provided document, RISC stands for Reduced Instruction Set Computer. It is a processor design approach that uses a comparatively small and simple set of instructions. Most RISC instructions are designed to perform one basic operation, such as adding two values, comparing values, or moving data between registers. 

Key features of RISC processors include:
* **Load/store design:** Arithmetic and logical operations usually work on data held in CPU registers, while separate load and store instructions transfer data between memory and registers.
* **Large number of general-purpose registers** and regular instruction formats, which help make instruction decoding and pipelining simpler.
* **Relative simplicity and regularity** in its instruction set rather than lacking power, where complex tasks are completed by combining several simple instructions.

USER:
What are its main advantages?

ASSISTANT:
Based on the pro

Step 18: Prepare the Streamlit Interface
Now we'll move from a notebook-based experiment to a proper chatbot interface.


What is Streamlit?
Streamlit is a Python framework that lets us build web applications without needing to write HTML, CSS, or JavaScrip

1. Install Streamlit

In [30]:

%pip install -q streamlit


2. Check the installation

In [31]:

import streamlit as st

print("Streamlit installed successfully!")
print("Streamlit version:", st.__version__)


Streamlit installed successfully!
Streamlit version: 1.67.0


3. Create the project directory
We'll organize the application instead of putting everything into one large Python file.

In [32]:

import os

project_structure = [
    "document_ai",
    "document_ai/src",
    "document_ai/data",
    "document_ai/chroma_db",
]

for directory in project_structure:
    os.makedirs(directory, exist_ok=True)

print("Project directories created successfully!")

for directory in project_structure:
    print(directory)


Project directories created successfully!
document_ai
document_ai/src
document_ai/data
document_ai/chroma_db


Step 19: Create document_loader.py

What is a document loader?
A document loader is responsible for reading files and extracting their contents.

In our project, it will:
1. Accept a PDF file path.
2. Open the PDF using PyPDF.
3. Extract text from every page.
4. Return the extracted text.
We'll use the PdfReader library that we already tested.

In [33]:

%%writefile document_ai/src/document_loader.py

from pypdf import PdfReader


def load_pdf(file_path: str) -> str:
    """
    Extract text from every page of a PDF file.
    """

    if not file_path:
        raise ValueError("PDF file path cannot be empty.")

    reader = PdfReader(file_path)

    if not reader.pages:
        raise ValueError("The PDF contains no pages.")

    extracted_text = ""

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""

        extracted_text += (
            f"\nPage {page_number}\n{page_text}"
        )

    if not extracted_text.strip():
        raise ValueError("No readable text found in the PDF.")

    return extracted_text


Writing document_ai/src/document_loader.py


In [34]:

import sys
import os

# Make the project directory available for imports
project_path = os.path.abspath("document_ai")

if project_path not in sys.path:
    sys.path.insert(0, project_path)

from src.document_loader import load_pdf

# Load the existing PDF
document_text = load_pdf(pdf_filename)

print("PDF loaded successfully!")
print("Extracted characters:", len(document_text))

print("\nFirst 500 characters:")
print(document_text[:500])


PDF loaded successfully!
Extracted characters: 13134

First 500 characters:

Page 1
Page 1
RISC AND CISC PROCESSORS
A  processor,  also  known  as  a  central  processing  unit  (CPU),  is  the  main  component  of  a 
computer that executes instructions and controls many of the system's operations. A processor  
uses an instruction set architecture (ISA), which defines the instructions that software can use  
and how those instructions are represented and executed by the hardware.
Two  important  approaches  to  processor  instruction-set  design  are  Reduced  Instruc


Step 20: Create text_splitter.py

What is a text splitter?
Remember when we divided the PDF's extracted text into chunks of 1,000 characters with an overlap of 200 characters?

Now we'll move that functionality into its own Python file.

Why do we need this?
- Large documents are difficult to process all at once.
- Smaller chunks make semantic search more effective.
- Overlap helps preserve information between neighboring chunks.

In [35]:

%%writefile document_ai/src/text_splitter.py

def split_text(
    text: str,
    chunk_size: int = 1000,
    overlap: int = 200
) -> list[str]:
    """
    Split text into smaller overlapping chunks.
    """

    if chunk_size <= 0:
        raise ValueError("Chunk size must be greater than zero.")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError(
            "Overlap must be non-negative and smaller than chunk size."
        )

    if not text or not text.strip():
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


Writing document_ai/src/text_splitter.py


In [36]:

from src.text_splitter import split_text

# Use the extracted PDF text
chunks = split_text(
    document_text,
    chunk_size=1000,
    overlap=200
)

print("Text splitting completed!")

print("Total characters:", len(document_text))
print("Total chunks:", len(chunks))

for index, chunk in enumerate(chunks[:3], start=1):
    print(f"\n--- Chunk {index} ---")
    print(chunk[:200])


Text splitting completed!
Total characters: 13134
Total chunks: 17

--- Chunk 1 ---
Page 1
Page 1
RISC AND CISC PROCESSORS
A  processor,  also  known  as  a  central  processing  unit  (CPU),  is  the  main  component  of  a 
computer that executes instructions and controls many of t

--- Chunk 2 ---
complex operations. [1], [2]
These architectures are important in Very Large Scale Integration (VLSI) because instruction-set  
design  influences  processor  datapaths,  control  logic,  registers,  

--- Chunk 3 ---
rk on data held in CPU registers. Separate load and  
store instructions transfer data between memory and registers. RISC designs also commonly use  
a large number of general-purpose registers and re


In [37]:

if len(chunks) >= 2:
    first_chunk_end = chunks[0][-200:]
    second_chunk_start = chunks[1][:200]

    print("End of first chunk:")
    print(first_chunk_end)

    print("\nStart of second chunk:")
    print(second_chunk_start)

    print("\nOverlap verification:")
    print("Note: strip() removes surrounding whitespace,")
    print("so compare the original text slices for exact overlap.")


End of first chunk:
  
complex operations. [1], [2]
These architectures are important in Very Large Scale Integration (VLSI) because instruction-set  
design  influences  processor  datapaths,  control  logic,  registers

Start of second chunk:
complex operations. [1], [2]
These architectures are important in Very Large Scale Integration (VLSI) because instruction-set  
design  influences  processor  datapaths,  control  logic,  registers,  

Overlap verification:
Note: strip() removes surrounding whitespace,
so compare the original text slices for exact overlap.


Step 21: Create embeddings.py

What are embeddings?
An embedding converts text into a numerical vector that represents its semantic meaning.

In [38]:

%%writefile document_ai/src/embeddings.py

from sentence_transformers import SentenceTransformer


MODEL_NAME = "all-MiniLM-L6-v2"

# Load the embedding model
embedding_model = SentenceTransformer(MODEL_NAME)


def generate_embeddings(texts: list[str]):
    """
    Convert a list of texts into numerical embeddings.
    """

    if not texts:
        raise ValueError("Text list cannot be empty.")

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    return embeddings


def generate_query_embedding(query: str):
    """
    Convert a single user question into an embedding.
    """

    if not query or not query.strip():
        raise ValueError("Query cannot be empty.")

    embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    return embedding


Writing document_ai/src/embeddings.py


In [39]:

from src.embeddings import (
    embedding_model,
    generate_embeddings,
    generate_query_embedding
)

# Generate embeddings for document chunks
chunk_embeddings = generate_embeddings(chunks)

print("\nDocument embeddings generated!")

print("Number of chunks:", len(chunks))
print("Embedding shape:", chunk_embeddings.shape)
print(
    "Vector dimensions:",
    embedding_model.get_sentence_embedding_dimension()
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Document embeddings generated!
Number of chunks: 17
Embedding shape: (17, 384)
Vector dimensions: 384


/tmp/ipykernel_1687/57590633.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [40]:

question = "What are the characteristics of RISC processors?"

query_embedding = generate_query_embedding(question)

print("Question embedding generated!")
print("Embedding shape:", query_embedding.shape)
print("First 10 values:")
print(query_embedding[0][:10])


Question embedding generated!
Embedding shape: (1, 384)
First 10 values:
[-0.03431882 -0.03781203 -0.00381657 -0.01825548 -0.00052207 -0.05291195
 -0.04119608  0.06332331 -0.01600923 -0.01037868]


Step 22: Create vector_store.py

What is a vector store?
A vector store is a database designed to store and search numerical embeddings.

In our project, ChromaDB stores:
- Document chunks.
- Their numerical embeddings.
- Unique IDs.
- Metadata such as the PDF filename and chunk index.
It also allows us to search for document chunks that are semantically similar to a user's question.

We'll move three responsibilities into vector_store.py:
1. Initialize ChromaDB.
2. Store document chunks and embeddings.
3. Search for relevant chunks.

In [41]:

%%writefile document_ai/src/vector_store.py

import chromadb

from src.embeddings import generate_query_embedding


# Initialize persistent ChromaDB client
chroma_client = chromadb.PersistentClient(
    path="./document_ai/chroma_db"
)

# Create or retrieve collection
collection = chroma_client.get_or_create_collection(
    name="document_collection",
    metadata={"hnsw:space": "cosine"}
)


def store_documents(
    chunks: list[str],
    embeddings,
    source: str
) -> None:
    """
    Store document chunks, embeddings, and metadata.
    """

    if not chunks:
        raise ValueError("No document chunks provided.")

    if len(chunks) != len(embeddings):
        raise ValueError(
            "Number of chunks and embeddings must match."
        )

    chunk_ids = [
        f"{source}_chunk_{i}"
        for i in range(len(chunks))
    ]

    metadata = [
        {
            "source": source,
            "chunk_index": i
        }
        for i in range(len(chunks))
    ]

    collection.upsert(
        ids=chunk_ids,
        documents=chunks,
        embeddings=embeddings.tolist(),
        metadatas=metadata
    )

    print(f"Successfully stored {len(chunks)} chunks!")


def search_documents(
    query: str,
    n_results: int = 3
) -> dict:
    """
    Retrieve relevant document chunks using semantic search.
    """

    if not query or not query.strip():
        raise ValueError("Search query cannot be empty.")

    if n_results <= 0:
        raise ValueError("n_results must be greater than zero.")

    total_documents = collection.count()

    if total_documents == 0:
        raise ValueError("No documents are stored in ChromaDB.")

    query_embedding = generate_query_embedding(query)

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=min(n_results, total_documents),
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    return results


Writing document_ai/src/vector_store.py


In [42]:

from src.vector_store import store_documents, collection

store_documents(
    chunks=chunks,
    embeddings=chunk_embeddings,
    source=os.path.basename(pdf_filename)
)

print("Total records in ChromaDB:", collection.count())


Successfully stored 17 chunks!
Total records in ChromaDB: 17


In [43]:

from src.vector_store import search_documents

question = "What are the main characteristics of RISC processors?"

results = search_documents(
    query=question,
    n_results=3
)

print("QUESTION:", question)

for index, document in enumerate(results["documents"][0]):
    print(f"\n--- Result {index + 1} ---")
    print("Metadata:", results["metadatas"][0][index])
    print("Distance:", results["distances"][0][index])
    print("Content:")
    print(document[:300])


QUESTION: What are the main characteristics of RISC processors?

--- Result 1 ---
Metadata: {'source': 'VLSI_Assignment_RISC_and_CISC_Processors.pdf', 'chunk_index': 14}
Distance: 0.30296391248703003
Content:
system design.
In VLSI design, the choice of architecture is one part of a larger engineering decision. Designers  
must also consider the target application, performance requirements, energy budget, silicon area,  
cost, software ecosystem, and compatibility.
8. Conclusion
RISC and CISC are two maj

--- Result 2 ---
Metadata: {'chunk_index': 1, 'source': 'VLSI_Assignment_RISC_and_CISC_Processors.pdf'}
Distance: 0.3066372871398926
Content:
complex operations. [1], [2]
These architectures are important in Very Large Scale Integration (VLSI) because instruction-set  
design  influences  processor  datapaths,  control  logic,  registers,  pipeline  design,  power  
consumption, performance, and the amount of hardware required on a chip.


--- Result 3 ---
Metadata: {'chunk_index': 2,

In [44]:

import src.vector_store as vector_store

# Delete the old collection
vector_store.chroma_client.delete_collection(
    name="document_collection"
)

# Create a clean collection
vector_store.collection = (
    vector_store.chroma_client.get_or_create_collection(
        name="document_collection",
        metadata={"hnsw:space": "cosine"}
    )
)

# Store only the current PDF chunks
vector_store.store_documents(
    chunks=chunks,
    embeddings=chunk_embeddings,
    source=os.path.basename(pdf_filename)
)

print("\nCollection successfully rebuilt!")
print("Total records:", vector_store.collection.count())
print("Expected records:", len(chunks))


Successfully stored 17 chunks!

Collection successfully rebuilt!
Total records: 17
Expected records: 17


In [45]:

results = vector_store.search_documents(
    query="What are the main characteristics of RISC processors?",
    n_results=3
)

for index, metadata in enumerate(results["metadatas"][0]):
    print(f"Result {index + 1}:")
    print(metadata)


Result 1:
{'source': 'VLSI_Assignment_RISC_and_CISC_Processors.pdf', 'chunk_index': 14}
Result 2:
{'source': 'VLSI_Assignment_RISC_and_CISC_Processors.pdf', 'chunk_index': 1}
Result 3:
{'source': 'VLSI_Assignment_RISC_and_CISC_Processors.pdf', 'chunk_index': 2}


Step 23: Create chatbot.py

What is the chatbot module?
So far, we have separated our application into individual modules:
Module	Responsibility
document_loader.py	Extract PDF text
text_splitter.py	Divide text into chunks
embeddings.py	Generate embeddings
vector_store.py	Store and retrieve document chunks
chatbot.py	Generate answers using Gemini

The chatbot.py module will connect ChromaDB and Gemini.

It will support:
- Context preparation.
- Normal answer generation.
- Streaming answer generation.
- Conversation history.

In [46]:

%%writefile document_ai/src/chatbot.py

from google import genai

from src.vector_store import search_documents


MODEL_NAME = "gemini-3.5-flash-lite"


def prepare_context(search_results: dict) -> str:
    """
    Combine retrieved document chunks into one context.
    """

    documents = search_results["documents"][0]

    if not documents:
        return "No relevant information was found."

    context_parts = []

    for index, document in enumerate(documents, start=1):
        context_parts.append(
            f"Document Chunk {index}:\n{document}"
        )

    return "\n\n".join(context_parts)


class DocumentChatbot:
    """
    Chatbot that answers questions using PDF documents.
    """

    def __init__(self, api_key: str):
        if not api_key:
            raise ValueError("Gemini API key is required.")

        self.client = genai.Client(api_key=api_key)

        self.conversation_history = []

    def _build_prompt(self, question: str, context: str) -> str:
        """
        Build the prompt using context and conversation history.
        """

        history = ""

        for message in self.conversation_history:
            history += (
                f"{message['role'].upper()}: "
                f"{message['content']}\n"
            )

        return f"""
You are a helpful PDF question-answering assistant.

Answer using ONLY the provided document context.
Use conversation history to understand follow-up questions.

If the answer is not available in the context,
clearly say that you could not find it.

Do not invent information.

CONVERSATION HISTORY:
{history}

DOCUMENT CONTEXT:
{context}

CURRENT QUESTION:
{question}

ANSWER:
"""

    def _retrieve_context(self, question: str, n_results: int):
        """
        Retrieve relevant chunks and prepare context.
        """

        search_results = search_documents(
            query=question,
            n_results=n_results
        )

        context = prepare_context(search_results)

        return context, search_results

    def _save_message(self, role: str, content: str):
        self.conversation_history.append({
            "role": role,
            "content": content
        })

    def ask(self, question: str, n_results: int = 3) -> dict:
        """
        Generate a complete answer.
        """

        if not question or not question.strip():
            raise ValueError("Question cannot be empty.")

        context, search_results = self._retrieve_context(
            question,
            n_results
        )

        prompt = self._build_prompt(question, context)

        response = self.client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        answer = response.text

        if not answer:
            raise RuntimeError("Gemini returned an empty response.")

        self._save_message("user", question)
        self._save_message("assistant", answer)

        return {
            "question": question,
            "answer": answer,
            "context": context,
            "sources": search_results["metadatas"][0]
        }

    def ask_stream(self, question: str, n_results: int = 3):
        """
        Generate and stream an answer.
        """

        if not question or not question.strip():
            raise ValueError("Question cannot be empty.")

        context, search_results = self._retrieve_context(
            question,
            n_results
        )

        prompt = self._build_prompt(question, context)

        response_stream = self.client.models.generate_content_stream(
            model=MODEL_NAME,
            contents=prompt
        )

        answer_parts = []

        for chunk in response_stream:
            if chunk.text:
                answer_parts.append(chunk.text)
                yield chunk.text

        answer = "".join(answer_parts)

        if not answer:
            raise RuntimeError("Gemini returned an empty response.")

        self._save_message("user", question)
        self._save_message("assistant", answer)

    def clear_history(self):
        """
        Clear the conversation history.
        """

        self.conversation_history.clear()


Writing document_ai/src/chatbot.py


In [47]:

from google.colab import userdata

from src.chatbot import DocumentChatbot

api_key = userdata.get("GEMINI_API_KEY")

chatbot = DocumentChatbot(api_key=api_key)

print("Document chatbot initialized successfully!")
print("Model:", "gemini-3.5-flash-lite")


Document chatbot initialized successfully!
Model: gemini-3.5-flash-lite


In [48]:

result = chatbot.ask(
    "What are the main characteristics of RISC processors?"
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)


QUESTION:
What are the main characteristics of RISC processors?

ANSWER:
Based on the provided document, the main characteristics of RISC (Reduced Instruction Set Computer) processors include:

* **Small and simple instruction set:** It uses a comparatively small and regular set of simple instructions, where most instructions are designed to perform one basic operation (such as adding two values, comparing values, or moving data between registers).
* **Load/store design:** Arithmetic and logical operations usually work on data held in CPU registers, while separate load and store instructions are used to transfer data between memory and registers.
* **Large number of general-purpose registers:** RISC designs commonly use many general-purpose registers.
* **Regular instruction formats:** This regularity makes instruction decoding and efficient pipelining simpler.

SOURCES:
{'source': 'VLSI_Assignment_RISC_and_CISC_Processors.pdf', 'chunk_index': 14}
{'source': 'VLSI_Assignment_RISC_and_C

In [49]:

print("QUESTION: What are its advantages?\n")
print("ANSWER:")

for text_chunk in chatbot.ask_stream(
    "What are its advantages?"
):
    print(text_chunk, end="", flush=True)

print("\n\nCONVERSATION MESSAGES:")
print(len(chatbot.conversation_history))


QUESTION: What are its advantages?

ANSWER:
Based on the provided document, the advantages of RISC processors include:

* **Simpler instruction decoding:** Regular instruction formats make the instruction-decoding stage easier to design.
* **Efficient pipelining:** Simple and predictable instructions help a processor divide work into pipeline stages.
* **Potential for lower power and area:** A simpler implementation may reduce hardware complexity, which is useful in power- and area-constrained devices.
* **Good compiler support:** A regular instruction set makes it easier for compilers to schedule and optimize instructions.
* **Scalability and flexibility:** RISC-based designs can be adapted for a wide range of applications, from small microcontrollers to high-performance processors.

CONVERSATION MESSAGES:
4


Step 24: Build the Streamlit interface

What will our application look like?
We'll create Document AI, with:
- A title and description.
- A sidebar for uploading PDF documents.
- A button to process uploaded documents.
- A chat interface for asking questions.
- Streaming Gemini responses.
- Conversation history.
- A button to clear the conversation.

In [50]:

%%writefile document_ai/app.py

import os
import tempfile

import streamlit as st

from src.document_loader import load_pdf
from src.text_splitter import split_text
from src.embeddings import generate_embeddings
from src.chatbot import DocumentChatbot

import src.vector_store as vector_store


# --------------------------------------------------
# Page configuration
# --------------------------------------------------

st.set_page_config(
    page_title="Document AI",
    page_icon="🤖",
    layout="wide"
)


# --------------------------------------------------
# Application title
# --------------------------------------------------

st.title("🤖 Document AI")
st.write("Upload a PDF and ask questions about its content.")


# --------------------------------------------------
# Initialize session state
# --------------------------------------------------

if "chatbot" not in st.session_state:
    st.session_state.chatbot = None

if "document_ready" not in st.session_state:
    st.session_state.document_ready = False


# --------------------------------------------------
# Sidebar: PDF upload
# --------------------------------------------------

with st.sidebar:
    st.header("📄 Document")

    uploaded_file = st.file_uploader(
        "Upload a PDF document",
        type=["pdf"]
    )

    process_button = st.button(
        "Process Document",
        disabled=uploaded_file is None,
        use_container_width=True
    )

    st.divider()

    if st.button(
        "Clear Conversation",
        use_container_width=True
    ):
        if st.session_state.chatbot:
            st.session_state.chatbot.clear_history()

        st.rerun()


# --------------------------------------------------
# Process uploaded document
# --------------------------------------------------

if process_button and uploaded_file is not None:

    api_key = os.getenv("GEMINI_API_KEY")

    if not api_key:
        st.error("GEMINI_API_KEY environment variable is missing.")

    else:
        with st.spinner("Processing your PDF..."):

            temp_path = None

            try:
                # Save uploaded PDF temporarily
                with tempfile.NamedTemporaryFile(
                    delete=False,
                    suffix=".pdf"
                ) as temp_file:

                    temp_file.write(uploaded_file.getvalue())
                    temp_path = temp_file.name

                # Extract text
                document_text = load_pdf(temp_path)

                # Split text into chunks
                chunks = split_text(
                    document_text,
                    chunk_size=1000,
                    overlap=200
                )

                if not chunks:
                    raise ValueError("No text chunks were created.")

                # Generate embeddings
                embeddings = generate_embeddings(chunks)

                # Reset the old collection
                vector_store.chroma_client.delete_collection(
                    name="document_collection"
                )

                vector_store.collection = (
                    vector_store.chroma_client.get_or_create_collection(
                        name="document_collection",
                        metadata={"hnsw:space": "cosine"}
                    )
                )

                # Store document chunks
                vector_store.store_documents(
                    chunks=chunks,
                    embeddings=embeddings,
                    source=uploaded_file.name
                )

                # Initialize chatbot
                st.session_state.chatbot = DocumentChatbot(
                    api_key=api_key
                )

                st.session_state.document_ready = True

                st.success(
                    f"Successfully processed {uploaded_file.name}!"
                )

                st.write(f"Total chunks: {len(chunks)}")

            except Exception as error:
                st.session_state.document_ready = False
                st.session_state.chatbot = None
                st.error(f"Document processing failed: {error}")

            finally:
                if temp_path and os.path.exists(temp_path):
                    os.remove(temp_path)


# --------------------------------------------------
# Display chat history
# --------------------------------------------------

if st.session_state.chatbot:

    for message in st.session_state.chatbot.conversation_history:

        with st.chat_message(message["role"]):
            st.markdown(message["content"])


# --------------------------------------------------
# Chat interface
# --------------------------------------------------

if st.session_state.document_ready:

    question = st.chat_input(
        "Ask a question about your document..."
    )

    if question:

        with st.chat_message("user"):
            st.markdown(question)

        with st.chat_message("assistant"):

            try:
                response = st.write_stream(
                    st.session_state.chatbot.ask_stream(question)
                )

            except Exception as error:
                st.error(f"Answer generation failed: {error}")

else:
    st.info(
        "Upload a PDF document using the sidebar "
        "and click Process Document to start chatting."
    )


Writing document_ai/app.py


In [51]:

%%writefile document_ai/requirements.txt

streamlit
chromadb
pypdf
sentence-transformers
google-genai
python-dotenv
pytest


Writing document_ai/requirements.txt


In [52]:

import os

for root, directories, files_list in os.walk("document_ai"):
    directories[:] = [
        directory for directory in directories
        if directory != "__pycache__"
    ]

    level = root.replace("document_ai", "").count(os.sep)

    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for filename in files_list:
        print(f"{indent}    {filename}")


document_ai/
    requirements.txt
    app.py
    src/
        vector_store.py
        document_loader.py
        embeddings.py
        text_splitter.py
        chatbot.py
    chroma_db/
        chroma.sqlite3
        63ea8fa5-aeef-4888-8d81-806f3698ef0e/
            data_level0.bin
            header.bin
            link_lists.bin
            length.bin
        92b7e1fe-8ce7-43e2-a0ca-e007b7407053/
            data_level0.bin
            header.bin
            link_lists.bin
            length.bin
    data/


Step 25: Configure the Gemini API key

In [54]:

import os
from google.colab import userdata

# Retrieve API key from Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY was not found in Colab Secrets.")

# Make it available to Streamlit
os.environ["GEMINI_API_KEY"] = api_key

print("Gemini API key configured successfully!")
print("API key is not displayed.")


Gemini API key configured successfully!
API key is not displayed.


Step 26: Check Streamlit installation

Before launching the application, let's make sure Streamlit is available.

In [55]:

import streamlit as st

print("Streamlit version:", st.__version__)
print("Application file exists:", os.path.exists("document_ai/app.py"))
print("Requirements file exists:", os.path.exists("document_ai/requirements.txt"))


Streamlit version: 1.67.0
Application file exists: True
Requirements file exists: True


Step 27: Launch the application

Streamlit normally runs on port 8501.

Google Colab doesn't automatically expose that local port in your browser, so we'll use Colab's built-in port forwardin

In [56]:

import subprocess
import time

# Start Streamlit from the notebook's current directory
process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "document_ai/app.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.address=0.0.0.0"
    ],
    stdout=open("/tmp/document_ai_streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

if process.poll() is None:
    print("Streamlit application started successfully!")
    print("Process ID:", process.pid)
else:
    print("Streamlit failed to start.")
    print(open("/tmp/document_ai_streamlit.log").read())


Streamlit application started successfully!
Process ID: 18626


In [57]:

from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=800)


<IPython.core.display.Javascript object>